# Member 3 – Leakage Investigation & Feature Engineering

**Author / Responsibility:** Member 3 – Leakage Investigation & Feature Engineering  
**Project:** Hotel Booking Cancellation Prediction (IT3051 Mini Project)  
**Branch:** `feature/it23842762-feature-engineering`

---

### Responsibilities of Member 3:
1. **Load / Continue from Cleaned Data:** Seamlessly continue from Member 2's cleaned baseline without redoing previous work.
2. **Target Variable Identification:** Clarify the prediction target (`is_canceled`).
3. **Data Leakage Investigation:** Systematically analyze all features for potential post-outcome target leakage and document justifications.
4. **Remove Leakage Columns:** Remove confirmed leakage attributes (`reservation_status`, `reservation_status_date`) while strictly preserving target `is_canceled`.
5. **Feature Engineering:** Create domain-informed, leak-free predictive features (`total_guests`, `total_stay`, `is_family`, `room_changed`, `has_special_requests`, `has_previous_cancellations`, `has_previous_bookings`, `is_weekend_only`).
6. **Feature Validation & Sanity Checks:** Comprehensive checks verifying non-negativity, missingness, datatypes, and distributions.
7. **Final Feature Set & Target Separation:** Formulate final feature matrix $X$ and target vector $y$.
8. **Handoff to Member 4:** Document exact output specifications for Member 4's encoding, scaling, and preprocessing pipeline.

## 1. Load / Continue from Cleaned Data

Member 1 (EDA & Problem Understanding) and Member 2 (Data Quality & Cleaning) have already completed their tasks.
- **Member 1** analyzed the raw dataset (119,390 rows, 32 columns) and target distribution.
- **Member 2** performed missing-value assessment, removed 31,994 exact duplicate rows, and eliminated 168 invalid records (zero-guest bookings and erroneous/negative ADR).
- **Cleaned Data Dimensions:** 87,228 rows and 32 columns.

We continue directly from Member 2's cleaned baseline by importing our modular pipeline from `src/feature_engineering/feature_engineering.py`.

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to Python path for reusable module import
sys.path.append(os.path.abspath("../../src"))

from feature_engineering.feature_engineering import (
    load_cleaned_data,
    identify_leakage_columns,
    remove_leakage_columns,
    create_features,
    validate_features,
    prepare_feature_target
)

# Set plotting aesthetics
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["font.size"] = 10

# Load Member 2 cleaned baseline dataset
DATA_PATH = "../../data/raw/hotel_bookings.csv"
df_clean = load_cleaned_data(DATA_PATH)

print(f"Cleaned Dataset Loaded Successfully!")
print(f"Dataset Shape: {df_clean.shape[0]:,} rows, {df_clean.shape[1]} columns")
df_clean.head()

Cleaned Dataset Loaded Successfully!
Dataset Shape: 87,228 rows, 32 columns


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


## 2. Target Variable

The target variable for our prediction model is:

$$\mathbf{y} = \text{is\_canceled}$$

- **`0`**: Booking was not canceled (guest checked in and completed stay).
- **`1`**: Booking was canceled (guest canceled prior to arrival or failed to show up).

Our objective is to train a machine learning model that predicts cancellation risk **at or before booking confirmation time**, using only features legitimately known prior to the stay outcome.

In [2]:
target_dist = df_clean["is_canceled"].value_counts(normalize=True) * 100
target_counts = df_clean["is_canceled"].value_counts()

print("Target Variable ('is_canceled') Distribution in Cleaned Dataset:")
for label, count, pct in zip(target_counts.index, target_counts.values, target_dist.values):
    status_str = "Not Canceled (0)" if label == 0 else "Canceled (1)"
    print(f"  - {status_str}: {count:,} bookings ({pct:.2f}%)")

Target Variable ('is_canceled') Distribution in Cleaned Dataset:
  - Not Canceled (0): 63,220 bookings (72.48%)
  - Canceled (1): 24,008 bookings (27.52%)


## 3. Data Leakage Investigation

### What is Data Leakage?
**Data leakage (or target leakage)** occurs when predictive models are trained on features that contain information about the target variable that would **not realistically be available** at the operational time when the model makes predictions in production. 

Including leakage features gives an illusion of near-perfect accuracy during model training and validation, but causes catastrophic failure when deployed on unseen operational bookings.

### Systematic Feature Audit & Justifications

| Feature Name | Investigation & Domain Meaning | Leakage Assessment | Action Taken |
| :--- | :--- | :--- | :--- |
| **`reservation_status`** | Records the final booking status (`Check-Out`, `Canceled`, `No-Show`). Recorded strictly after the booking lifecycle concludes. | **Definite Post-Outcome Target Leakage.** Has a 100% deterministic mathematical relationship with `is_canceled` (`Check-Out` $\rightarrow 0$, `Canceled`/`No-Show` $\rightarrow 1$). | **Remove from feature set.** |
| **`reservation_status_date`** | Records the exact date on which the reservation status changed (e.g. date of cancellation or checkout date). | **Definite Post-Outcome Target Leakage.** This timestamp is generated after the cancellation/checkout event takes place. | **Remove from feature set.** |
| **`days_in_waiting_list`** | Number of days the booking spent on the waiting list before confirmation. | **Safe to Keep.** Known immediately upon booking confirmation prior to check-in. | **Retain in feature set.** |
| **`booking_changes`** | Number of amendments made to the booking between creation and check-in. | **Safe to Keep.** Accumulated up to the moment of prediction. | **Retain in feature set.** |
| **`deposit_type`** | Deposit policy agreed at reservation time (`No Deposit`, `Non Refund`, `Refundable`). | **Safe to Keep.** Determined at time of reservation. | **Retain in feature set.** |
| **`lead_time`** | Number of days between booking creation date and arrival date. | **Safe to Keep.** Fundamental pre-arrival reservation attribute. | **Retain in feature set.** |

In [3]:
# Retrieve leakage investigation details
leakage_info = identify_leakage_columns()
leakage_columns = leakage_info["confirmed_leakage"]

print("Confirmed Data Leakage Columns to Remove:")
for col in leakage_columns:
    print(f"  [LEAKAGE DETECTED] -> {col}")
    print(f"    Reason: {leakage_info['justifications'][col]}\n")

# Verify deterministic correlation with reservation_status
print("Deterministic Mapping between reservation_status and is_canceled:")
display(pd.crosstab(df_clean["reservation_status"], df_clean["is_canceled"], margins=True))

Confirmed Data Leakage Columns to Remove:
  [LEAKAGE DETECTED] -> reservation_status
    Reason: Direct post-outcome target leakage. Contains final booking states ('Check-Out', 'Canceled', 'No-Show') which are 100% deterministically correlated with the target variable is_canceled.

  [LEAKAGE DETECTED] -> reservation_status_date
    Reason: Post-outcome target leakage. Represents the date when the reservation status was last updated (e.g., exact cancellation date or checkout date), which is unknown at booking time prior to cancellation.

Deterministic Mapping between reservation_status and is_canceled:


is_canceled,0,1,All
reservation_status,,,
Canceled,0,22995,22995
Check-Out,63220,0,63220
No-Show,0,1013,1013
All,63220,24008,87228


## 4. Remove Leakage Features

We remove only the confirmed leakage columns (`reservation_status` and `reservation_status_date`).  
*Note: We verify that the target column `is_canceled` is preserved in the DataFrame.*

In [4]:
# Remove confirmed leakage features
df_no_leakage = remove_leakage_columns(df_clean, leakage_columns)

print(f"Shape before leakage removal: {df_clean.shape}")
print(f"Shape after leakage removal:  {df_no_leakage.shape}")
print(f"Columns removed ({len(leakage_columns)}): {leakage_columns}")
print(f"Is target 'is_canceled' present? {'is_canceled' in df_no_leakage.columns}")

Shape before leakage removal: (87228, 32)
Shape after leakage removal:  (87228, 30)
Columns removed (2): ['reservation_status', 'reservation_status_date']
Is target 'is_canceled' present? True


## 5. Feature Engineering

To enhance the model's predictive capacity, we construct meaningful, domain-specific, leak-free features from existing raw attributes.

### 5.1 `total_guests`
- **Formula:** $\text{total\_guests} = \text{adults} + \text{children} + \text{babies}$
- **Domain Rationale:** Captures total party size occupying the room. Large groups may exhibit distinct booking commitment behaviors compared to solo travelers.
- **Handling Missing Values:** Four missing entries in `children` are safely treated as $0$ during addition.

### 5.2 `total_stay`
- **Formula:** $\text{total\_stay} = \text{stays\_in\_week\_nights} + \text{stays\_in\_weekend\_nights}$
- **Domain Rationale:** Represents the complete length of stay in nights. Longer stays represent higher financial commitment and may carry different cancellation likelihoods.

### 5.3 `is_family`
- **Formula:** $\text{is\_family} = 1 \text{ if } (\text{children} + \text{babies} > 0) \text{ else } 0$
- **Domain Rationale:** Family bookings have distinct planning horizons, seasonal vacation commitments, and lower flexibility to cancel.

### 5.4 Additional Justified Features
1. **`room_changed`**: $1 \text{ if } (\text{reserved\_room\_type} \neq \text{assigned\_room\_type}) \text{ else } 0$. Operational room reassignment/upgrade indicator.
2. **`has_special_requests`**: $1 \text{ if } (\text{total\_of\_special\_requests} > 0) \text{ else } 0$. Signals higher engagement and intent to stay.
3. **`has_previous_cancellations`**: $1 \text{ if } (\text{previous\_cancellations} > 0) \text{ else } 0$. Captures historical cancellation propensity.
4. **`has_previous_bookings`**: $1 \text{ if } (\text{previous\_bookings\_not\_canceled} > 0) \text{ else } 0$. Indicator of established customer loyalty.
5. **`is_weekend_only`**: $1 \text{ if } (\text{stays\_in\_weekend\_nights} > 0 \text{ and } \text{stays\_in\_week\_nights} == 0) \text{ else } 0$. Captures short weekend getaway trips.

In [5]:
# Apply feature engineering
df_engineered = create_features(df_no_leakage)

engineered_feature_names = [
    "total_guests",
    "total_stay",
    "is_family",
    "room_changed",
    "has_special_requests",
    "has_previous_cancellations",
    "has_previous_bookings",
    "is_weekend_only"
]

print(f"Shape before feature engineering: {df_no_leakage.shape}")
print(f"Shape after feature engineering:  {df_engineered.shape}")
print(f"Number of new features created:   {len(engineered_feature_names)}")
df_engineered[engineered_feature_names].head(10)

Shape before feature engineering: (87228, 30)
Shape after feature engineering:  (87228, 38)
Number of new features created:   8


,total_guests,total_stay,is_family,room_changed,has_special_requests,has_previous_cancellations,has_previous_bookings,is_weekend_only
0,2,0,0,0,0,0,0,0
1,2,0,0,0,0,0,0,0
2,1,1,0,1,0,0,0,0
3,1,1,0,0,0,0,0,0
4,2,2,0,0,1,0,0,0
6,2,2,0,0,0,0,0,0
7,2,2,0,0,1,0,0,0
8,2,3,0,0,1,0,0,0
9,2,3,0,0,0,0,0,0
10,2,4,0,0,0,0,0,0


## 6. Feature Validation & Sanity Checks

We execute validation checks on all newly engineered features:
1. **Data Types:** Verified as clean integer types (`int64` / `int32`).
2. **Missing Values:** Ensure 0 missing values across all new features.
3. **Domain Value Bounds:**
   - `total_guests`: Minimum is $\ge 1$ (no ghost zero-guest stays).
   - `total_stay`: Minimum is $\ge 0$ (no negative night stays).
   - Binary features (`is_family`, `room_changed`, etc.): Values strictly bounded to $\{0, 1\}$.

In [6]:
# Generate validation summary table
val_summary = validate_features(df_no_leakage, df_engineered, engineered_feature_names)
display(val_summary)

# Save validation summary table
val_summary.to_csv("../../results/feature_engineering/tables/feature_validation_summary.csv", index=False)
print("Validation table saved to results/feature_engineering/tables/feature_validation_summary.csv")

,Feature Name,Data Type,Missing Count,Min Value,Max Value,Unique Values Count,Sample Values
0,total_guests,int64,0,1,55,14,"[np.int64(2), np.int64(1), np.int64(3), np.int..."
1,total_stay,int64,0,0,69,42,"[np.int64(0), np.int64(1), np.int64(2), np.int..."
2,is_family,int64,0,0,1,2,"[np.int64(0), np.int64(1)]"
3,room_changed,int64,0,0,1,2,"[np.int64(0), np.int64(1)]"
4,has_special_requests,int64,0,0,1,2,"[np.int64(0), np.int64(1)]"
5,has_previous_cancellations,int64,0,0,1,2,"[np.int64(0), np.int64(1)]"
6,has_previous_bookings,int64,0,0,1,2,"[np.int64(0), np.int64(1)]"
7,is_weekend_only,int64,0,0,1,2,"[np.int64(0), np.int64(1)]"


Validation table saved to results/feature_engineering/tables/feature_validation_summary.csv


In [7]:
# Sanity check assertions
assert (df_engineered["total_guests"] >= 1).all(), "Sanity Check Failed: Negative or zero total guests found!"
assert (df_engineered["total_stay"] >= 0).all(), "Sanity Check Failed: Negative total stay found!"
for col in ["is_family", "room_changed", "has_special_requests", "has_previous_cancellations", "has_previous_bookings", "is_weekend_only"]:
    assert set(df_engineered[col].unique()).issubset({0, 1}), f"Sanity Check Failed: Binary feature {col} contains non-binary values!"
assert df_engineered[engineered_feature_names].isnull().sum().sum() == 0, "Sanity Check Failed: Missing values detected in new features!"

print("ALL SANITY CHECKS PASSED SUCCESSFULLY!")

ALL SANITY CHECKS PASSED SUCCESSFULLY!


## 7. Final Feature Set & Target Separation

We separate the prepared dataset into:
- **$X$ (Feature Matrix):** Contains all unencoded, unscaled predictor columns including raw valid features and newly engineered features.
- **$y$ (Target Vector):** Series containing the binary ground truth `is_canceled`.

In [8]:
# Separate feature matrix X and target vector y
X, y = prepare_feature_target(df_engineered, target_col="is_canceled")

print("==================================================")
print("             FINAL DATASET DIMENSIONS             ")
print("==================================================")
print(f"X (Feature Matrix) Shape: {X.shape[0]:,} rows, {X.shape[1]} features")
print(f"y (Target Vector) Shape:  {y.shape[0]:,} rows")
print(f"Target 'is_canceled' in X: {'is_canceled' in X.columns}")
print(f"Leakage columns in X:     {any(c in X.columns for c in leakage_columns)}")
print("==================================================")

             FINAL DATASET DIMENSIONS             
X (Feature Matrix) Shape: 87,228 rows, 37 features
y (Target Vector) Shape:  87,228 rows
Target 'is_canceled' in X: False
Leakage columns in X:     False


In [9]:
# Summary breakdown of feature types in X
num_features = X.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X.select_dtypes(include=["object", "str"]).columns.tolist()

print(f"Total Features in X: {len(X.columns)}")
print(f"  - Numerical Features ({len(num_features)}): {num_features}")
print(f"  - Categorical Features ({len(cat_features)}): {cat_features}")

Total Features in X: 37
  - Numerical Features (27): ['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'agent', 'company', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'total_guests', 'total_stay', 'is_family', 'room_changed', 'has_special_requests', 'has_previous_cancellations', 'has_previous_bookings', 'is_weekend_only']
  - Categorical Features (10): ['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type']


## 8. Handoff to Member 4

Member 3's responsibilities are now fully accomplished. 

### Output Summary for Member 4:
1. **Clean Feature Matrix ($X$):**
   - Shape: `(87,228, 37)`
   - Contains all raw valid features and 8 engineered features (`total_guests`, `total_stay`, `is_family`, `room_changed`, `has_special_requests`, `has_previous_cancellations`, `has_previous_bookings`, `is_weekend_only`).
   - Zero target leakage columns.
2. **Target Vector ($y$):**
   - Shape: `(87,228,)`
   - Clean binary series `is_canceled`.
3. **Pre-processing State:**
   - Categorical and numerical columns remain unencoded and unscaled.
   - Missing values in `company` (94.31%), `agent` (13.69%), `country` (0.41%), and `children` (0.003%) are preserved for Member 4's imputation and `ColumnTransformer` preprocessing pipeline.
   - Stratified train/test split, scaling, encoding, and modeling belong strictly to Member 4 and subsequent members.